In [ ]:
#  CELL 0 : Output capture — run FIRST
import os, sys
OUT_DIR = "ps4_pyspark_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream
        self._log = log_fh
        self._is_tee = True
    def write(self, data):
        for s in (self._jupyter, self._log):
            try: s.write(data); s.flush()
            except Exception: pass
    def flush(self):
        for s in (self._jupyter, self._log):
            try: s.flush()
            except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] tee -> ps4_pyspark_outputs/console.log")


**PS4 — Hourly Anomaly Detection (PySpark)**

Full PySpark port of `PS4_SageMaker_MLflow_FeatureStore.ipynb` — separate cells for signals, EDA, SPC, PCA anomaly, station analytics, Prophet.

**Run order:** 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15 → 16 → 17 → 18 → 19

**S3 outputs:** `s3://{artifact-bucket}/chicago/ps4/scored/asof=<date>/device_weekly/` (primary, device×ISO-week) + `anomalies/` / `timeline/` / `outliers/` (weekly) + `device_daily/` (kept for PS1 joins) + `.../artifacts/run_date=<date>/`


In [ ]:
#  CELL 1 : Install
import subprocess, sys
pkgs = [
    "pyspark>=3.4", "s3fs", "pyarrow", "scikit-learn>=1.4", "scipy",
    "prophet", "mlflow>=2.13", "sagemaker-mlflow",
    "boto3", "sagemaker>=2.220,<3.0", "pandas>=2.0", "matplotlib", "seaborn", "joblib",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("OK")


In [ ]:
#  CELL 2 : Imports
import os, json, time, shutil, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3, mlflow, joblib
import mlflow.spark
from scipy import stats
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline as SparkPipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import f1_score, precision_recall_curve, roc_curve

try:
    from sagemaker import Session, get_execution_role
except ImportError:
    from sagemaker.session import Session, get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum

warnings.filterwarnings("ignore")
print("Imports OK")


In [ ]:
#  CELL 3 : Configuration
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps4_hourly"
GOLD_S3 = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION = Session()
ROLE = get_execution_role()
REGION = boto3.Session().region_name
SM_CLIENT = boto3.client("sagemaker", region_name=REGION)

MLFLOW_SERVER_ARN = (
    "arn:aws:sagemaker:us-east-1:170202974600:"
    "mlflow-tracking-server/cubic-mars-mlflow-server-dev"
)
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)

EXPERIMENT_NAME = "chicago-ps4-anomaly-detection"
FS_GROUP_NAME = "chicago-ps4-features"
TARGET = "ensemble_anomaly_flag"
TARGET_COL = TARGET
SIGNAL_THRESHOLD = 2
DEVICE_TYPES = ["TVM", "GATE", "VALIDATOR"]
DEV_COL = "mars_device_category"

CKPT_DIR = "/home/sagemaker-user/PS4/checkpoints"
SPARK_CKPT_VERSION = "ps4-pyspark-v2"
SPARK_CKPT_META = os.path.join(CKPT_DIR, "spark_ckpt_meta.joblib")
SPARK_CKPT_SPLITS = os.path.join(CKPT_DIR, "spark_splits")
LOCAL_SPARK_TEMP_DIR = os.path.join(CKPT_DIR, "spark_tmp")
CKPT_WRITE_PARTITIONS = 16
HADOOP_AWS_PACKAGE = "org.apache.hadoop:hadoop-aws:3.3.4"
AWS_SDK_BUNDLE = "com.amazonaws:aws-java-sdk-bundle:1.12.262"
SPARK_JARS_PACKAGES = f"{HADOOP_AWS_PACKAGE},{AWS_SDK_BUNDLE}"
FORCE_RELOAD = False
CHECKPOINT_LOADED = False
CHECKPOINT_STAGE = None
ENABLE_FEATURE_STORE = False
VIZ_MAX_ROWS = 250_000

COMMON_TAGS = {
    "ps": "PS4", "target": TARGET, "city": "chicago",
    "pipeline": "pyspark", "architecture": "CUBIC-MARS-AWS-v6",
    "gold_version": "None",
}


CITY_CODE = "CHI"
PS4_S3_PREFIX = "chicago/ps4"
PS4_S3_BASE = f"s3://{ARTIFACT_BUCKET}/{PS4_S3_PREFIX}"
PS4_SCORED_S3 = f"{PS4_S3_BASE}/scored"
PS4_ARTIFACTS_S3 = f"{PS4_S3_BASE}/artifacts"
ENABLE_S3_EXPORT = True
PS4_S3_EXPORT_GRAIN = "device_weekly"  # CELL 19 aggregates hourly -> device-week; also writes device_daily for PS1

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")
print(f"PS4 output S3: {PS4_S3_BASE}")


In [ ]:
#  CELL 4 : Spark session + S3A + checkpoint helpers
import sys
from pyspark.sql import SparkSession

AUTO_INSTALL_LOCAL_JAVA = True
OPENJDK_VERSION = "17"
LOCAL_SPARK_THREADS = "4"
LOCAL_SPARK_DRIVER_MEMORY = "16g"
LOCAL_SPARK_DRIVER_MEMORY_GB = 16
LOCAL_SHUFFLE_PARTITIONS = 32
spark = None
SPARK_RUNTIME = "unknown"
shuffle_partitions = LOCAL_SHUFFLE_PARTITIONS
spc_results = {}
anomaly_days = {}

def _run_conda(requirements):
    try:
        ip = get_ipython()  # noqa: F821
    except NameError:
        ip = None
    if ip is not None:
        ip.run_line_magic("conda", f"install -y -c conda-forge {' '.join(requirements)}")
    else:
        ce = shutil.which("conda")
        if not ce:
            raise RuntimeError(f"Run: %conda install -y -c conda-forge openjdk={OPENJDK_VERSION}")
        subprocess.check_call([ce, "install", "-y", "-c", "conda-forge", *requirements])


def _java_from_home(java_home):
    if not java_home:
        return None
    c = os.path.join(java_home, "bin", "java")
    return c if os.path.isfile(c) and os.access(c, os.X_OK) else None


def _discover_java():
    ch = os.environ.get("JAVA_HOME")
    cj = _java_from_home(ch)
    if cj:
        return os.path.realpath(ch), os.path.realpath(cj)
    for c in [shutil.which("java"), os.path.join(sys.prefix, "bin", "java"),
              "/usr/lib/jvm/java-17-amazon-corretto/bin/java",
              "/usr/lib/jvm/java-11-amazon-corretto/bin/java"]:
        if c and os.path.isfile(c) and os.access(c, os.X_OK):
            rj = os.path.realpath(c)
            rh = os.path.dirname(os.path.dirname(rj))
            if _java_from_home(rh):
                return rh, rj
    return None, None


def ensure_local_java():
    jh, je = _discover_java()
    if je is None and AUTO_INSTALL_LOCAL_JAVA:
        print(f"Java absent — installing OpenJDK {OPENJDK_VERSION} ...")
        _run_conda([f"openjdk={OPENJDK_VERSION}"])
        jh, je = _discover_java()
    if je is None:
        raise RuntimeError(f"Install Java: %conda install -y -c conda-forge openjdk={OPENJDK_VERSION}")
    os.environ["JAVA_HOME"] = jh
    jb = os.path.join(jh, "bin")
    if jb not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = jb + os.pathsep + os.environ.get("PATH", "")
    print(f"JAVA_HOME: {jh}")
    return jh


def _apply_s3a_hadoop_conf(_spark):
    """Override S3A duration defaults like '60s' that break Hadoop 3.3.x parsers."""
    _s3a_numeric = {
        "fs.s3a.connection.timeout": "200000",
        "fs.s3a.connection.establish.timeout": "30000",
        "fs.s3a.connection.request.timeout": "60000",
        "fs.s3a.connection.acquisition.timeout": "60000",
        "fs.s3a.connection.idle.time": "60000",
        "fs.s3a.connection.ttl": "300000",
        "fs.s3a.socket.timeout": "60000",
        "fs.s3a.threads.keepalivetime": "60",
        "fs.s3a.multipart.purge.age": "86400000",
    }
    for key, value in _s3a_numeric.items():
        try:
            _spark.conf.set(f"spark.hadoop.{key}", value)
        except Exception as exc:
            print(f"[S3A] spark.conf {key}: {type(exc).__name__}")
    try:
        _hconf = _spark.sparkContext._jsc.hadoopConfiguration()
        for key, value in _s3a_numeric.items():
            _hconf.set(key, value)
        _spark._jvm.org.apache.hadoop.fs.FileSystem.closeAll()
    except Exception as exc:
        print(f"[S3A] hadoopConfiguration override: {type(exc).__name__}")


def _apply_spark_runtime_conf(_spark):
    global SPARK_RUNTIME, shuffle_partitions
    SPARK_RUNTIME = "local" if _spark.sparkContext.master.startswith("local") else "managed"
    shuffle_partitions = LOCAL_SHUFFLE_PARTITIONS
    for k, v in {
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.parquet.filterPushdown": "true",
        "spark.sql.shuffle.partitions": str(shuffle_partitions),
        "spark.sql.execution.arrow.pyspark.enabled": "false",
        "spark.sql.session.timeZone": "UTC",
        "spark.memory.fraction": "0.6",
        "spark.memory.storageFraction": "0.3",
        "spark.sql.autoBroadcastJoinThreshold": "10485760",
    }.items():
        try:
            _spark.conf.set(k, v)
        except Exception:
            pass
    _apply_s3a_hadoop_conf(_spark)
    _spark.sparkContext.setLogLevel("WARN")


def spark_path(uri):
    return "s3a://" + uri[len("s3://"):] if uri.startswith("s3://") else uri


def read_parquet_s3(path):
    ensure_spark_alive()
    _apply_s3a_hadoop_conf(spark)
    p = spark_path(path)
    print(f"[S3A] Reading {p}")
    return spark.read.parquet(p)


def build_spark():
    ensure_local_java()
    os.makedirs(LOCAL_SPARK_TEMP_DIR, exist_ok=True)
    esa = os.environ.get("PYSPARK_SUBMIT_ARGS", "pyspark-shell")
    if "--driver-memory" not in esa:
        if not esa.strip().endswith("pyspark-shell"):
            esa = esa.strip() + " pyspark-shell"
        os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {LOCAL_SPARK_DRIVER_MEMORY} {esa}"
    sess = (
        SparkSession.builder.appName("PS4-PySpark-Anomaly")
        .master(f"local[{LOCAL_SPARK_THREADS}]")
        .config("spark.driver.memory", LOCAL_SPARK_DRIVER_MEMORY)
        .config("spark.driver.maxResultSize", f"{max(1, LOCAL_SPARK_DRIVER_MEMORY_GB // 2)}g")
        .config("spark.local.dir", LOCAL_SPARK_TEMP_DIR)
        .config("spark.jars.packages", SPARK_JARS_PACKAGES)
        .config("spark.python.worker.memory", "512m")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.timeout", "200000")
        .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000")
        .config("spark.hadoop.fs.s3a.connection.request.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.acquisition.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.idle.time", "60000")
        .config("spark.hadoop.fs.s3a.connection.ttl", "300000")
        .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
        .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60")
        .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400000")
        .getOrCreate()
    )
    _apply_spark_runtime_conf(sess)
    return sess


def ensure_spark_alive(recreate=False):
    global spark
    _s = globals().get("spark")
    if _s is not None and not recreate:
        try:
            _s.range(1).count()
            return _s
        except Exception as e:
            print(f"[WARN] Spark dead ({type(e).__name__}) — recreating")
    if recreate or _s is not None:
        try:
            a = SparkSession.getActiveSession()
            if a:
                a.stop()
        except Exception:
            pass
    spark = build_spark()
    return spark


def _save_ckpt_meta(meta, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    joblib.dump(meta, path)


def _load_ckpt_meta(path):
    return joblib.load(path) if os.path.exists(path) else None


def _checkpoint_parquet_ok(path):
    """True when checkpoint dir contains at least one readable parquet part file."""
    if not os.path.isdir(path):
        return False
    for root, _, files in os.walk(path):
        if any(f.endswith(".parquet") for f in files):
            return True
    return False


def clear_spark_checkpoint():
    """Remove corrupt/partial checkpoint artifacts."""
    if os.path.isdir(SPARK_CKPT_SPLITS):
        shutil.rmtree(SPARK_CKPT_SPLITS, ignore_errors=True)
    if os.path.isfile(SPARK_CKPT_META):
        os.remove(SPARK_CKPT_META)
    print(f"[CHECKPOINT] Cleared {SPARK_CKPT_SPLITS}")


def _prune_for_checkpoint(sdf):
    """Drop ML intermediate columns before persisting — reduces spill + disk use."""
    _drop = {
        "if_pca", "if_raw", "if_scaled", "features_raw", "features", "pca_features",
        "pca2", "pca_full", "_if_arr", "_date_tmp", "_hw_sum",
    }
    _drop |= {c for c in sdf.columns if c.endswith("__imp") or c.endswith("__if")}
    _drop = [c for c in _drop if c in sdf.columns]
    return sdf.drop(*_drop) if _drop else sdf


def save_spark_etl_checkpoint(stage="etl"):
    global CHECKPOINT_STAGE, df_spark, train_sdf, val_sdf, test_sdf, FEATURE_COLS, IF_FEATURE_COLS
    ensure_spark_alive()
    os.makedirs(CKPT_DIR, exist_ok=True)
    os.makedirs(LOCAL_SPARK_TEMP_DIR, exist_ok=True)
    try:
        if hasattr(df_spark, "is_cached") and df_spark.is_cached:
            df_spark.unpersist()
    except Exception:
        pass
    _to_write = _prune_for_checkpoint(df_spark)
    _parts = CKPT_WRITE_PARTITIONS
    print(f"[CHECKPOINT] Writing stage={stage}  partitions={_parts}  cols={len(_to_write.columns)}")
    (
        _to_write.coalesce(_parts)
        .write.mode("overwrite")
        .option("compression", "snappy")
        .partitionBy("split")
        .parquet(SPARK_CKPT_SPLITS)
    )
    if not _checkpoint_parquet_ok(SPARK_CKPT_SPLITS):
        clear_spark_checkpoint()
        raise RuntimeError(
            f"Checkpoint write produced no parquet files under {SPARK_CKPT_SPLITS}. "
            "Try lowering CKPT_WRITE_PARTITIONS or freeing disk space."
        )
    df_spark = spark.read.parquet(SPARK_CKPT_SPLITS)
    train_sdf = df_spark.filter(F.col("split") == "train")
    val_sdf = df_spark.filter(F.col("split") == "val")
    test_sdf = df_spark.filter(F.col("split") == "test")
    _avail = set(df_spark.columns)
    FEATURE_COLS = [c for c in FEATURE_COLS if c in _avail]
    IF_FEATURE_COLS = [c for c in IF_FEATURE_COLS if c in _avail] or FEATURE_COLS[:15]
    meta = {
        "version": SPARK_CKPT_VERSION, "stage": stage,
        "FEATURE_COLS": list(FEATURE_COLS), "IF_FEATURE_COLS": list(IF_FEATURE_COLS),
        "SIGNAL_COLS": list(SIGNAL_COLS), "TARGET": TARGET, "DEV_COL": DEV_COL,
        "DEVICE_TYPES": DEVICE_TYPES, "SCALE_POS_WEIGHT": SCALE_POS_WEIGHT,
        "spc_results": spc_results,
    }
    _save_ckpt_meta(meta, SPARK_CKPT_META)
    CHECKPOINT_STAGE = stage
    print(f"[CHECKPOINT] Saved stage={stage} -> {SPARK_CKPT_SPLITS}")


def restore_spark_checkpoint():
    global df_spark, train_sdf, val_sdf, test_sdf, FEATURE_COLS, IF_FEATURE_COLS
    global SIGNAL_COLS, SCALE_POS_WEIGHT, CHECKPOINT_LOADED, CHECKPOINT_STAGE
    global spc_results
    meta = _load_ckpt_meta(SPARK_CKPT_META)
    if meta is None or meta.get("version") != SPARK_CKPT_VERSION:
        return False
    if not _checkpoint_parquet_ok(SPARK_CKPT_SPLITS):
        print(f"[CHECKPOINT] Missing/corrupt parquet at {SPARK_CKPT_SPLITS} — reloading from Gold")
        clear_spark_checkpoint()
        return False
    ensure_spark_alive()
    try:
        df_spark = spark.read.parquet(SPARK_CKPT_SPLITS)
        df_spark.limit(1).count()  # validate schema/read
    except Exception as exc:
        print(f"[CHECKPOINT] Read failed ({type(exc).__name__}: {exc}) — reloading from Gold")
        clear_spark_checkpoint()
        return False
    train_sdf = df_spark.filter(F.col("split") == "train")
    val_sdf = df_spark.filter(F.col("split") == "val")
    test_sdf = df_spark.filter(F.col("split") == "test")
    FEATURE_COLS = list(meta["FEATURE_COLS"])
    IF_FEATURE_COLS = list(meta.get("IF_FEATURE_COLS", FEATURE_COLS[:15]))
    _avail = set(df_spark.columns)
    FEATURE_COLS = [c for c in FEATURE_COLS if c in _avail]
    IF_FEATURE_COLS = [c for c in IF_FEATURE_COLS if c in _avail] or FEATURE_COLS[:15]
    SIGNAL_COLS = list(meta["SIGNAL_COLS"])
    SCALE_POS_WEIGHT = meta["SCALE_POS_WEIGHT"]
    spc_results = meta.get("spc_results", {})
    CHECKPOINT_STAGE = meta.get("stage", "etl")
    CHECKPOINT_LOADED = True
    print(f"[CHECKPOINT] Restored stage={CHECKPOINT_STAGE}")
    return True




def write_parquet_s3(sdf, uri, partition_cols=None, mode="overwrite"):
    """Write a Spark DataFrame to S3 as Snappy Parquet (s3a://)."""
    ensure_spark_alive()
    _apply_s3a_hadoop_conf(spark)
    p = spark_path(uri.rstrip("/") + "/")
    writer = (
        sdf.coalesce(CKPT_WRITE_PARTITIONS)
        .write.mode(mode)
        .option("compression", "snappy")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(p)
    print(f"[S3A] Wrote parquet -> {p}")
    return p


def upload_file_to_s3(local_path, s3_key, bucket=None):
    """Upload a single local file to the artifacts bucket."""
    import mimetypes
    bucket = bucket or ARTIFACT_BUCKET
    ctype = mimetypes.guess_type(local_path)[0] or "application/octet-stream"
    boto3.client("s3", region_name=REGION).upload_file(
        local_path, bucket, s3_key, ExtraArgs={"ContentType": ctype},
    )
    uri = f"s3://{bucket}/{s3_key}"
    print(f"[S3] Uploaded {os.path.basename(local_path)} -> {uri}")
    return uri


def upload_out_dir_to_s3(local_dir=None, run_date=None):
    """Upload png/json/log files from OUT_DIR to chicago/ps4/artifacts/run_date=..."""
    local_dir = local_dir or OUT_DIR
    run_date = run_date or pd.Timestamp.utcnow().strftime("%Y-%m-%d")
    uploaded = []
    for root, _, files in os.walk(local_dir):
        for fn in files:
            local_path = os.path.join(root, fn)
            rel = os.path.relpath(local_path, local_dir).replace("\\", "/")
            key = f"{PS4_S3_PREFIX}/artifacts/run_date={run_date}/{rel}"
            uploaded.append(upload_file_to_s3(local_path, key))
    print(f"[S3] {len(uploaded)} artifact file(s) under s3://{ARTIFACT_BUCKET}/{PS4_S3_PREFIX}/artifacts/run_date={run_date}/")
    return uploaded

spark = ensure_spark_alive()
print(f"Spark {spark.version}  master={spark.sparkContext.master}")


In [ ]:
#  CELL 5 : Load Gold (PySpark Parquet on S3)
if FORCE_RELOAD:
    clear_spark_checkpoint()
if not FORCE_RELOAD and restore_spark_checkpoint():
    print("[CHECKPOINT] Skipping Gold load — restored splits")
else:
    print(f"Loading Gold: {GOLD_S3}")
    df_spark = read_parquet_s3(GOLD_S3)
    if "hour_dt" not in df_spark.columns and "hour_bucket" in df_spark.columns:
        df_spark = df_spark.withColumnRenamed("hour_bucket", "hour_dt")
    df_spark = df_spark.withColumn("hour_dt", F.to_timestamp("hour_dt"))
    GOLD_VERSION = None
    DATA_SOURCE = "pyspark_parquet"
    print(f"Rows: {df_spark.count():,}")
    df_spark.groupBy(TARGET).count().show()
    _rate = df_spark.agg(F.mean(TARGET).alias("r")).collect()[0]["r"]
    print(f"Anomaly rate: {_rate:.2%}")


In [ ]:
#  CELL 6 : Build signal ensemble flags (PySpark)
if CHECKPOINT_LOADED:
    print("[CHECKPOINT] Signals already in restored dataframe")
else:
    _dev = DEV_COL if DEV_COL in df_spark.columns else next(
        (c for c in ["device_type", "device_category", "mars_device_category"] if c in df_spark.columns), None)
    if _dev:
        DEV_COL = _dev
        df_spark = df_spark.withColumn(DEV_COL, F.upper(F.trim(F.col(DEV_COL))))

    if "event_count_hour" in df_spark.columns and "transit_day" in df_spark.columns:
        _daily = (
            df_spark.groupBy("DEVICE_ID", "transit_day")
            .agg(F.sum("event_count_hour").alias("daily_event_count"))
        )
        if "daily_event_count" in df_spark.columns:
            df_spark = df_spark.drop("daily_event_count")
        df_spark = df_spark.join(_daily, on=["DEVICE_ID", "transit_day"], how="left")

    # Signal 1: SPC
    if "event_rate_anomaly" in df_spark.columns:
        df_spark = df_spark.withColumn("signal_spc", F.coalesce(F.col("event_rate_anomaly"), F.lit(0)).cast("int"))
    elif all(c in df_spark.columns for c in ["event_count_hour", "baseline_mean_events", "baseline_stddev_events"]):
        df_spark = df_spark.withColumn(
            "signal_spc",
            (F.abs(F.col("event_count_hour") - F.col("baseline_mean_events"))
             > 3 * F.coalesce(F.col("baseline_stddev_events"), F.lit(1))).cast("int"))
    else:
        df_spark = df_spark.withColumn("signal_spc", F.lit(0))

    df_spark = df_spark.withColumn("signal_isolation_forest", F.lit(0))

    # Signal 3: tap rejection
    if "tap_reject_rate_daily" in df_spark.columns:
        df_spark = df_spark.withColumn("signal_tap_rejection", (F.col("tap_reject_rate_daily") > 5.0).cast("int"))
    elif "reject_rate_anomaly" in df_spark.columns:
        df_spark = df_spark.withColumn("signal_tap_rejection", F.coalesce(F.col("reject_rate_anomaly"), F.lit(0)).cast("int"))
    else:
        df_spark = df_spark.withColumn("signal_tap_rejection", F.lit(0))

    # Signal 4: metric / latency
    if "metric_anomaly" in df_spark.columns:
        df_spark = df_spark.withColumn("signal_metric", F.coalesce(F.col("metric_anomaly"), F.lit(0)).cast("int"))
    elif all(c in df_spark.columns for c in ["metric_401_avg_ms_hour", "baseline_mean_metric401", "baseline_stddev_metric401"]):
        df_spark = df_spark.withColumn(
            "signal_metric",
            (F.abs(F.col("metric_401_avg_ms_hour") - F.col("baseline_mean_metric401"))
             > 2 * F.coalesce(F.col("baseline_stddev_metric401"), F.lit(1))).cast("int"))
    else:
        df_spark = df_spark.withColumn("signal_metric", F.lit(0))

    # Signal 5: revenue zero
    if "use_revenue_zero_flag" in df_spark.columns:
        df_spark = df_spark.withColumn("signal_revenue_zero", F.coalesce(F.col("use_revenue_zero_flag"), F.lit(0)).cast("int"))
    else:
        df_spark = df_spark.withColumn("signal_revenue_zero", F.lit(0))

    # Signal 6: TVM HW
    _hw = [c for c in ["bhu_count_hour", "chu_count_hour", "printer_count_hour"] if c in df_spark.columns]
    if _hw:
        df_spark = df_spark.withColumn("_hw_sum", sum(F.coalesce(F.col(c), F.lit(0)) for c in _hw))
        _p90 = df_spark.filter((F.col(DEV_COL) == "TVM") & (F.col("_hw_sum") > 0)).approxQuantile("_hw_sum", [0.90], 0.01)
        _thr = float(_p90[0]) if _p90 else 1.0
        df_spark = df_spark.withColumn("signal_tvm_hw", (F.col("_hw_sum") >= F.lit(_thr)).cast("int")).drop("_hw_sum")
    else:
        df_spark = df_spark.withColumn("signal_tvm_hw", F.lit(0))

    # Signal 7: OOS
    if "oos_count_hour" in df_spark.columns:
        _p75 = df_spark.filter(F.col("oos_count_hour") > 0).approxQuantile("oos_count_hour", [0.75], 0.01)
        _thr = float(_p75[0]) if _p75 else 1.0
        df_spark = df_spark.withColumn("signal_oos", (F.col("oos_count_hour") >= F.lit(_thr)).cast("int"))
    else:
        df_spark = df_spark.withColumn("signal_oos", F.lit(0))

    SIGNAL_COLS = [c for c in [
        "signal_spc", "signal_isolation_forest", "signal_tap_rejection",
        "signal_metric", "signal_revenue_zero", "signal_tvm_hw", "signal_oos",
    ] if c in df_spark.columns]

    df_spark = df_spark.withColumn("signal_active_count", sum(F.col(c) for c in SIGNAL_COLS))
    if TARGET not in df_spark.columns:
        df_spark = df_spark.withColumn(TARGET, (F.col("signal_active_count") >= F.lit(SIGNAL_THRESHOLD)).cast("int"))

    # Summary table (Spark agg)
    _SRC_MAP = {
        "signal_spc": "event_rate_anomaly (Gold 28d baseline)",
        "signal_isolation_forest": "Spark PCA anomaly — filled CELL 14",
        "signal_tap_rejection": "tap_reject_rate_daily > 5%",
        "signal_metric": "metric_anomaly / latency spike",
        "signal_revenue_zero": "use_revenue_zero_flag",
        "signal_tvm_hw": "BHU+CHU+printer >= p90 TVM non-zero hrs",
        "signal_oos": "oos_count_hour >= p75 non-zero hrs",
    }
    _rows = []
    for s in SIGNAL_COLS:
        _r = df_spark.agg(F.mean(s).alias("rate"), F.sum(s).alias("n")).collect()[0]
        row = {"Signal": s, "Source": _SRC_MAP.get(s, s),
               "Rate_All": f"{_r['rate']:.2%}", "Active_Rows": int(_r["n"] or 0)}
        if DEV_COL in df_spark.columns:
            _dev_rates = (
                df_spark.groupBy(DEV_COL).agg(F.mean(s).alias("rate")).collect()
            )
            for dr in _dev_rates:
                row[f"Rate_{dr[DEV_COL]}"] = f"{dr['rate']:.2%}"
        _rows.append(row)
    _tr = df_spark.agg(F.mean(TARGET).alias("rate"), F.sum(TARGET).alias("n")).collect()[0]
    _rows.append({"Signal": TARGET, "Source": "ensemble (2+ signals)",
                  "Rate_All": f"{_tr['rate']:.2%}", "Active_Rows": int(_tr["n"] or 0)})
    print("\nSignal Summary:")
    print(pd.DataFrame(_rows).to_string(index=False))
    if DEV_COL in df_spark.columns:
        df_spark.groupBy(DEV_COL).agg(
            F.count("*").alias("rows"), F.mean(TARGET).alias("anomaly_rate")
        ).orderBy(F.desc("anomaly_rate")).show()


In [ ]:
#  CELL 7 : Feature engineering & temporal splits (PySpark)
if CHECKPOINT_LOADED:
    print("[CHECKPOINT] Features/splits restored")
else:
    RAW_SIGNAL_COLS = ["event_rate_anomaly", "metric_anomaly", "reject_rate_anomaly"]
    _exclude = {TARGET, "DEVICE_ID", "hour_dt", "split", "signal_active_count",
                *SIGNAL_COLS, *RAW_SIGNAL_COLS, "daily_event_count"}
    FEATURE_COLS = sorted({
        f.name for f in df_spark.schema.fields
        if f.name not in _exclude and f.dataType.simpleString() in (
            "int", "bigint", "double", "float", "decimal", "smallint", "tinyint")
    })
    IF_FEATURE_COLS = [c for c in FEATURE_COLS if any(
        k in c.lower() for k in ["metric", "response", "rate", "count", "kpi", "percent"]
    )] or FEATURE_COLS[:15]

    df_spark = df_spark.withColumn(
        "split",
        F.when(F.col("hour_dt") < "2025-10-01", "train")
         .when(F.col("hour_dt") < "2026-01-01", "val")
         .otherwise("test"),
    )
    train_sdf = df_spark.filter(F.col("split") == "train")
    val_sdf = df_spark.filter(F.col("split") == "val")
    test_sdf = df_spark.filter(F.col("split") == "test")

    _pn = train_sdf.groupBy(TARGET).count().collect()
    _pos = next((r["count"] for r in _pn if r[TARGET] == 1), 0)
    _neg = next((r["count"] for r in _pn if r[TARGET] == 0), 0)
    SCALE_POS_WEIGHT = round(_neg / max(_pos, 1), 2)

    print(f"Features: {len(FEATURE_COLS)}  IF features: {len(IF_FEATURE_COLS)}")
    print(f"Train/val/test: {train_sdf.count():,} / {val_sdf.count():,} / {test_sdf.count():,}")
    print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")
    save_spark_etl_checkpoint(stage="etl")


In [ ]:
#  CELL 8 : Signal activation charts (Spark sample → pandas viz)
def spark_to_pandas(sdf=None, cols=None, max_rows=VIZ_MAX_ROWS):
    sdf = sdf if sdf is not None else df_spark
    if cols:
        sdf = sdf.select(*[c for c in cols if c in sdf.columns])
    n = sdf.count()
    if n > max_rows:
        sdf = sdf.sample(withReplacement=False, fraction=min(1.0, max_rows / max(n, 1)), seed=42)
    return sdf.toPandas()

df = spark_to_pandas(cols=SIGNAL_COLS + [TARGET, DEV_COL, "hour_dt"])

_sig_rates = {s: float(df[s].mean()) for s in SIGNAL_COLS if s in df.columns}
_sig_rates[TARGET] = float(df[TARGET].mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_sr = pd.Series(_sig_rates).sort_values()
_colors = ["#E53935" if k == TARGET else "#1565C0" for k in _sr.index]
_sr.plot(kind="barh", ax=axes[0], color=_colors, edgecolor="white")
axes[0].axvline(0.05, color="orange", ls="--", lw=1.2, label="5% ref")
axes[0].set_title("Signal activation rates (all rows)", fontweight="bold")
axes[0].legend()

if DEV_COL in df.columns:
    _hmap = df.groupby(DEV_COL)[SIGNAL_COLS + [TARGET]].mean()
    sns.heatmap(_hmap.T, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".1%", linewidths=0.5)
    axes[1].set_title("Signal rate by device type", fontweight="bold")

_active = df[SIGNAL_COLS].astype(float)
_co = _active.T.dot(_active)
_co_pct = _co.div(_co.values.diagonal(), axis=0)
np.fill_diagonal(_co_pct.values, np.nan)
sns.heatmap(_co_pct, ax=axes[2], cmap="Blues", annot=True, fmt=".0%",
            linewidths=0.5, mask=_co_pct.isna())
axes[2].set_title("Signal co-activation matrix", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "signal_activation.png"), dpi=100)
plt.show()
with mlflow.start_run(run_name="signal_activation_viz"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact(os.path.join(OUT_DIR, "signal_activation.png"))
print("Signal activation charts logged.")


In [ ]:
#  CELL 9 : EDA — full 2×3 panel (Spark agg + sample)
df_eda = spark_to_pandas(cols=SIGNAL_COLS + [TARGET, DEV_COL, "hour_dt"])
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

if len(SIGNAL_COLS) >= 2:
    sns.heatmap(df_eda[SIGNAL_COLS].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, ax=axes[0, 0], linewidths=0.5)
    axes[0, 0].set_title("Signal Co-activation Correlation")

df_eda[TARGET].value_counts().plot(kind="bar", ax=axes[0, 1], color=["#43A047", "#E53935"])
axes[0, 1].set_title("ensemble_anomaly_flag distribution")

df_eda["hour_dt"] = pd.to_datetime(df_eda["hour_dt"])
_hod = df_eda["hour_dt"].dt.hour
_dow = df_eda["hour_dt"].dt.day_name()
_DOW = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
_pivot = df_eda.assign(hour=_hod, dow=_dow).groupby(["dow", "hour"])[TARGET].mean().unstack(fill_value=0)
_pivot = _pivot.reindex([d for d in _DOW if d in _pivot.index])
sns.heatmap(_pivot, ax=axes[0, 2], cmap="YlOrRd", fmt=".0%", annot=True, linewidths=0.3)
axes[0, 2].set_title("Anomaly rate: hour × day-of-week")

_monthly = df_eda.assign(month=df_eda["hour_dt"].dt.to_period("M")).groupby("month")[TARGET].mean()
_monthly.index = _monthly.index.astype(str)
axes[1, 0].plot(_monthly.index, _monthly.values, marker="o", color="#1565C0")
axes[1, 0].set_title("Monthly anomaly rate trend")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].axhline(_monthly.mean(), color="red", ls="--", label="mean")
axes[1, 0].legend()

df_eda.groupby(_hod)[TARGET].mean().plot(kind="bar", ax=axes[1, 1], color="#1565C0")
axes[1, 1].set_title("Anomaly rate by hour of day")

if DEV_COL in df_eda.columns:
    df_eda.groupby(DEV_COL)[TARGET].mean().sort_values(ascending=False).plot(
        kind="bar", ax=axes[1, 2], color="#7B1FA2")
    axes[1, 2].set_title("Anomaly rate by device type")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "eda_full.png"), dpi=100)
plt.show()
with mlflow.start_run(run_name="eda_pyspark_full"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact(os.path.join(OUT_DIR, "eda_full.png"))


In [ ]:
#  CELL 10 : Feature Store (optional)
if not ENABLE_FEATURE_STORE:
    print("[SKIP] Feature Store disabled — set ENABLE_FEATURE_STORE=True")
else:
    _cutoff = train_sdf.agg(F.max("hour_dt")).collect()[0][0]
    fs_pdf = (
        train_sdf.filter(F.col("hour_dt") >= F.expr(f"timestamp '{_cutoff}' - interval 90 days"))
        .select(*FEATURE_COLS, TARGET, "DEVICE_ID", "hour_dt").limit(500_000).toPandas()
    )
    fs_pdf["event_time"] = pd.to_datetime(fs_pdf["hour_dt"]).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_pdf["DEVICE_ID"] = fs_pdf["DEVICE_ID"].astype(str)
    fs_pdf = fs_pdf.drop(columns=["hour_dt"])
    fg = FeatureGroup(name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
        feature_definitions=[FeatureDefinition(
            feature_name=c,
            feature_type=FeatureTypeEnum.FRACTIONAL if fs_pdf[c].dtype.kind == "f" else FeatureTypeEnum.INTEGRAL,
        ) for c in fs_pdf.columns])
    try:
        SM_CLIENT.delete_feature_group(FeatureGroupName=FS_GROUP_NAME)
        time.sleep(60)
    except Exception:
        pass
    fg.create(s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
              record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
              role_arn=ROLE, enable_online_store=True)
    fg.ingest(data_frame=fs_pdf.reset_index(drop=True), max_workers=8, wait=True)
    print(f"Ingested {len(fs_pdf):,} rows")


In [ ]:
#  CELL 11 : Ensemble helpers + split sync
from pyspark.ml import Pipeline

def recompute_ensemble(use_fleet_spc=False):
    global df_spark, train_sdf, val_sdf, test_sdf, SCALE_POS_WEIGHT
    _active_cols = []
    for c in SIGNAL_COLS:
        if c == "signal_spc" and use_fleet_spc and "signal_spc_fleet" in df_spark.columns:
            _active_cols.append(F.col("signal_spc_fleet"))
        else:
            _active_cols.append(F.col(c))
    df_spark = df_spark.withColumn("signal_active_count", sum(_active_cols))
    df_spark = df_spark.withColumn(TARGET, (F.col("signal_active_count") >= F.lit(SIGNAL_THRESHOLD)).cast("int"))
    train_sdf = df_spark.filter(F.col("split") == "train")
    val_sdf = df_spark.filter(F.col("split") == "val")
    test_sdf = df_spark.filter(F.col("split") == "test")
    _pn = train_sdf.groupBy(TARGET).count().collect()
    _pos = next((r["count"] for r in _pn if r[TARGET] == 1), 0)
    _neg = next((r["count"] for r in _pn if r[TARGET] == 0), 0)
    SCALE_POS_WEIGHT = round(_neg / max(_pos, 1), 2)
    return SCALE_POS_WEIGHT


def sync_splits_and_features(verbose=True):
    """Refresh train/val/test and drop FEATURE_COLS missing from df_spark."""
    global train_sdf, val_sdf, test_sdf, FEATURE_COLS, IF_FEATURE_COLS
    train_sdf = df_spark.filter(F.col("split") == "train")
    val_sdf = df_spark.filter(F.col("split") == "val")
    test_sdf = df_spark.filter(F.col("split") == "test")
    _avail = set(df_spark.columns)
    _missing = [c for c in FEATURE_COLS if c not in _avail]
    if _missing:
        if verbose:
            suffix = "..." if len(_missing) > 8 else ""
            print(f"[SYNC] Dropping {len(_missing)} FEATURE_COLS not in dataframe: {_missing[:8]}{suffix}")
        FEATURE_COLS = [c for c in FEATURE_COLS if c in _avail]
    IF_FEATURE_COLS = [c for c in IF_FEATURE_COLS if c in _avail] or FEATURE_COLS[:15]
    if not FEATURE_COLS:
        raise RuntimeError("No FEATURE_COLS remain after sync — re-run ETL/signal cells or FORCE_RELOAD=True")
    return train_sdf, val_sdf, test_sdf

print("Helpers ready")


In [ ]:
#  CELL 12 : SPC Signal 1 analysis + Option A fleet signal (PySpark)
ensure_spark_alive()
spc_results = {}
anomaly_days = {}

if "daily_event_count" in df_spark.columns and DEV_COL in df_spark.columns:
    for _dt in DEVICE_TYPES:
        _sub = df_spark.filter(F.col(DEV_COL) == _dt)
        if _sub.limit(1).count() == 0:
            continue
        _stats = _sub.agg(
            F.mean("baseline_mean_events").alias("mean"),
            F.mean("baseline_stddev_events").alias("std"),
        ).collect()[0]
        _mn = float(_stats["mean"] or 0)
        _sd = float(_stats["std"] or 1)
        spc_results[_dt] = {"mean": round(_mn, 4), "ucl": round(_mn + 3 * _sd, 4),
                            "lcl": max(0, round(_mn - 3 * _sd, 4))}

        _daily = (
            _sub.withColumn("date", F.date_format("hour_dt", "yyyy-MM-dd"))
            .groupBy("date")
            .agg(
                F.sum("signal_spc").alias("violation_device_hours"),
                F.mean("daily_event_count").alias("avg_event_count"),
            )
            .filter(F.col("violation_device_hours") > 0)
            .orderBy(F.desc("violation_device_hours"))
        )
        anomaly_days[_dt] = _daily.limit(500).toPandas()
        if len(anomaly_days[_dt]):
            anomaly_days[_dt]["device_type"] = _dt
            anomaly_days[_dt]["month"] = pd.to_datetime(anomaly_days[_dt]["date"]).dt.strftime("%Y-%m")

    with mlflow.start_run(run_name="spc_signal1_analysis"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "spc_control_chart", "signal": "S1"})
        for dt, vals in spc_results.items():
            for k, v in vals.items():
                mlflow.log_metric(f"{dt}_{k}", v)
            if dt in anomaly_days:
                mlflow.log_metric(f"{dt}_anomalous_days", len(anomaly_days[dt]))
        with open(os.path.join(OUT_DIR, "spc_thresholds.json"), "w") as f:
            json.dump(spc_results, f, indent=2)
        mlflow.log_artifact(os.path.join(OUT_DIR, "spc_thresholds.json"))

    print("SPC thresholds:", spc_results)
    for _dt, _adf in anomaly_days.items():
        if len(_adf):
            print(f"\n  {_dt} top violation days:")
            print(_adf.head(5).to_string(index=False))

# Top devices by SPC violations
_dev_col = "DEVICE_ID" if "DEVICE_ID" in df_spark.columns else None
if _dev_col and DEV_COL in df_spark.columns:
    for _dt in DEVICE_TYPES:
        _top = (
            df_spark.filter(F.col(DEV_COL) == _dt)
            .groupBy(_dev_col).agg(F.sum("signal_spc").alias("violation_device_hours"))
            .orderBy(F.desc("violation_device_hours")).limit(5).toPandas()
        )
        if len(_top):
            print(f"\nTop 5 {_dt} devices by SPC violations:")
            print(_top.to_string(index=False))

# Option A: fleet-level SPC (>5% fleet device-hours)
if all(c in df_spark.columns for c in ["signal_spc", DEV_COL, "hour_dt"]):
    df_spark = df_spark.withColumn("_date_tmp", F.date_format("hour_dt", "yyyy-MM-dd"))
    df_spark = df_spark.withColumn(
        "fleet_spc_rate",
        F.avg("signal_spc").over(Window.partitionBy(DEV_COL, "_date_tmp")),
    )
    df_spark = df_spark.withColumn("signal_spc_fleet", (F.col("fleet_spc_rate") > 0.05).cast("int"))
    df_spark = df_spark.drop("_date_tmp")
    if "fleet_spc_rate" not in FEATURE_COLS:
        FEATURE_COLS = sorted(set(FEATURE_COLS + ["fleet_spc_rate"]))
    _orig = df_spark.agg(F.mean("signal_spc")).collect()[0][0]
    _new = df_spark.agg(F.mean("signal_spc_fleet")).collect()[0][0]
    print(f"\nOption A fleet SPC: per-device rate={_orig:.1%} -> fleet rate={_new:.1%}")
    # Swap signal_spc for ensemble (Option A fix)
    df_spark = df_spark.withColumn("signal_spc", F.col("signal_spc_fleet"))
    SCALE_POS_WEIGHT = recompute_ensemble(use_fleet_spc=False)
    print(f"Ensemble updated after Option A. scale_pos_weight={SCALE_POS_WEIGHT}")
    with mlflow.start_run(run_name="spc_option_a_fleet_signal"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "spc_fleet", "signal": "S1"})
        mlflow.log_metric("orig_spc_rate", float(_orig))
        mlflow.log_metric("fleet_spc_rate", float(_new))
        mlflow.log_metric("scale_pos_weight", SCALE_POS_WEIGHT)

# Monthly heatmap
if anomaly_days:
    _all = pd.concat([v for v in anomaly_days.values() if len(v)], ignore_index=True)
    if "month" in _all.columns and len(_all):
        _pivot = _all.groupby(["device_type", "month"])["violation_device_hours"].sum().unstack(fill_value=0)
        fig, ax = plt.subplots(figsize=(max(10, len(_pivot.columns) * 0.8), 4))
        sns.heatmap(_pivot, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax)
        ax.set_title("SPC Violation Device-Hours — Monthly Heatmap")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "spc_heatmap.png"), dpi=100)
        plt.show()
        with mlflow.start_run(run_name="spc_monthly_heatmap"):
            mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
            mlflow.log_artifact(os.path.join(OUT_DIR, "spc_heatmap.png"))


In [ ]:
#  CELL 13 : SPC control charts + 7-day signal timeline
if spc_results and "daily_event_count" in df_spark.columns:
    _pdf_daily = (
        df_spark.filter(F.col(DEV_COL).isin(DEVICE_TYPES))
        .withColumn("date", F.date_format("hour_dt", "yyyy-MM-dd"))
        .groupBy(DEV_COL, "date")
        .agg(F.mean("daily_event_count").alias("daily_event_count"),
             F.sum("signal_spc").alias("violations"))
        .orderBy("date").toPandas()
    )
    fig, axes = plt.subplots(len(DEVICE_TYPES), 1, figsize=(14, 4 * len(DEVICE_TYPES)), squeeze=False)
    for i, _dt in enumerate(DEVICE_TYPES):
        if _dt not in spc_results:
            continue
        _sub = _pdf_daily[_pdf_daily[DEV_COL] == _dt]
        _v = spc_results[_dt]
        ax = axes[i, 0]
        ax.plot(_sub["date"], _sub["daily_event_count"], alpha=0.5, lw=0.8, label="daily events")
        ax.axhline(_v["ucl"], color="red", ls="--", label=f"UCL={_v['ucl']:.1f}")
        ax.axhline(_v["mean"], color="green", ls="-", label=f"mean={_v['mean']:.1f}")
        ax.axhline(_v["lcl"], color="orange", ls="--", label=f"LCL={_v['lcl']:.1f}")
        _viol = _sub[_sub["violations"] > 0]
        if len(_viol):
            ax.scatter(_viol["date"], _viol["daily_event_count"], color="red", s=10, zorder=5)
        ax.set_title(f"SPC Control Chart — {_dt}", fontweight="bold")
        ax.legend(fontsize=8)
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "spc_control_charts.png"), dpi=100)
    plt.show()
    with mlflow.start_run(run_name="spc_control_chart"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        mlflow.log_artifact(os.path.join(OUT_DIR, "spc_control_charts.png"))

# 7-day rolling signal timeline
_time_sigs = [s for s in SIGNAL_COLS + [TARGET] if s in df_spark.columns]
_ts = (
    df_spark.withColumn("day", F.date_format("hour_dt", "yyyy-MM-dd"))
    .groupBy("day").agg(*[F.mean(c).alias(c) for c in _time_sigs])
    .orderBy("day").toPandas()
)
_ts["day"] = pd.to_datetime(_ts["day"])
for c in _time_sigs:
    _ts[c] = _ts[c].rolling(7, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(18, 5))
for s in _time_sigs:
    ax.plot(_ts["day"], _ts[s], label=s, lw=2.5 if s == TARGET else 1.2,
            ls="-" if s == TARGET else "--")
ax.set_title("Signal activation — 7-day rolling average", fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "signal_timeline.png"), dpi=100)
plt.show()
with mlflow.start_run(run_name="signal_timeline"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact(os.path.join(OUT_DIR, "signal_timeline.png"))
if spc_results:
    print("\nSPC Thresholds:")
    print(pd.DataFrame(spc_results).T.round(2).to_string())


In [ ]:
#  CELL 14 : Spark PCA anomaly signal (replaces sklearn IF)
ensure_spark_alive()
_anom_path = os.path.join(CKPT_DIR, "pca_anomaly")

if os.path.isdir(_anom_path) and not FORCE_RELOAD:
    _anom_pipe = PipelineModel.load(_anom_path)
    print(f"[CHECKPOINT] Loaded PCA anomaly pipe from {_anom_path}")
else:
    _imputer = Imputer(inputCols=IF_FEATURE_COLS, outputCols=[f"{c}__if" for c in IF_FEATURE_COLS], strategy="median")
    _if_imp = [f"{c}__if" for c in IF_FEATURE_COLS]
    _assembler = VectorAssembler(inputCols=_if_imp, outputCol="if_raw", handleInvalid="keep")
    _scaler = StandardScaler(inputCol="if_raw", outputCol="if_scaled", withStd=True, withMean=True)
    _pca = PCA(k=min(10, len(IF_FEATURE_COLS)), inputCol="if_scaled", outputCol="if_pca")
    _anom_pipe = SparkPipeline(stages=[_imputer, _assembler, _scaler, _pca]).fit(train_sdf)
    os.makedirs(CKPT_DIR, exist_ok=True)
    _anom_pipe.write().overwrite().save(_anom_path)

# Native Spark L2 on PCA vector — avoids Python UDF memory blow-up on full dataset
_if_imp_cols = [f"{c}__if" for c in IF_FEATURE_COLS]
_scored = (
    _anom_pipe.transform(df_spark)
    .withColumn("_if_arr", vector_to_array("if_pca"))
    .withColumn("if_score", F.sqrt(F.expr("aggregate(_if_arr, 0D, (acc, x) -> acc + x * x)")))
    .drop("_if_arr", "if_pca", "if_raw", "if_scaled", *_if_imp_cols)
)
_contam = float(min(0.40, max(0.01, train_sdf.agg(F.mean(TARGET)).collect()[0][0] or 0.05)))
_thr = _scored.select("if_score").approxQuantile("if_score", [1.0 - _contam], 0.01)[0]
print(f"Spark PCA anomaly threshold (top {100*_contam:.1f}%): {_thr:.4f}")

df_spark = (
    _scored
    .withColumn("signal_isolation_forest", (F.col("if_score") >= F.lit(_thr)).cast("int"))
)
SCALE_POS_WEIGHT = recompute_ensemble()
if "if_score" not in FEATURE_COLS:
    FEATURE_COLS = sorted(set(FEATURE_COLS + ["if_score"]))

with mlflow.start_run(run_name="spark_pca_anomaly_signal2"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "spark_pca_anomaly", "signal": "S2"})
    mlflow.log_param("pca_k", min(10, len(IF_FEATURE_COLS)))
    mlflow.log_param("contamination", _contam)
    mlflow.log_metric("if_threshold", float(_thr))
    mlflow.log_metric("if_anomaly_rate_test", float(test_sdf.agg(F.mean("signal_isolation_forest")).collect()[0][0]))
    mlflow.spark.log_model(_anom_pipe, "pca_anomaly_pipe")

# Device-type IF breakdown
if DEV_COL in df_spark.columns:
    print("\nIsolation Forest (Spark PCA) — anomaly rate by device type:")
    for _split, _sdf in [("Train", train_sdf), ("Val", val_sdf), ("Test", test_sdf)]:
        _rates = _sdf.groupBy(DEV_COL).agg(F.mean("signal_isolation_forest").alias("rate")).collect()
        _parts = "  ".join(f"{r[DEV_COL]}={r['rate']:.1%}" for r in _rates)
        print(f"  {_split:5s}: {_parts}")

_dev_col = "DEVICE_ID" if "DEVICE_ID" in df_spark.columns else None
if _dev_col:
    print("\nTop 5 devices by IF rate (test):")
    print(
        test_sdf.groupBy(_dev_col)
        .agg(F.mean("signal_isolation_forest").alias("if_rate"), F.count("*").alias("hours"))
        .orderBy(F.desc("if_rate")).limit(5).toPandas().to_string(index=False)
    )
save_spark_etl_checkpoint(stage="signals")


In [ ]:
#  CELL 15 : PCA / IF diagnostic charts
_test_pdf = test_sdf.select("if_score", TARGET, "signal_isolation_forest").toPandas()
_train_pdf = train_sdf.select("if_score").toPandas()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].hist(_train_pdf["if_score"], bins=50, alpha=0.7, label="train", color="#1565C0")
axes[0, 0].hist(_test_pdf["if_score"], bins=50, alpha=0.5, label="test", color="#E53935")
axes[0, 0].axvline(_thr, color="black", ls="--", label=f"threshold={_thr:.3f}")
axes[0, 0].set_title("IF score distribution (Spark PCA L2)")
axes[0, 0].legend()

if _test_pdf[TARGET].nunique() > 1:
    _prec, _rec, _ = precision_recall_curve(_test_pdf[TARGET], _test_pdf["if_score"])
    axes[0, 1].plot(_rec, _prec, color="#1565C0")
    axes[0, 1].set_title("PR curve (IF score vs ensemble)")
    _fpr, _tpr, _ = roc_curve(_test_pdf[TARGET], _test_pdf["if_score"])
    axes[0, 2].plot(_fpr, _tpr, color="#1565C0")
    axes[0, 2].set_title("ROC curve")

if DEV_COL in test_sdf.columns:
    _dev_rates = test_sdf.groupBy(DEV_COL).agg(F.mean("signal_isolation_forest").alias("rate")).toPandas()
    axes[1, 0].bar(_dev_rates[DEV_COL], _dev_rates["rate"], color="#7B1FA2")
    axes[1, 0].set_title("IF anomaly rate by device (test)")

axes[1, 1].bar(["Normal", "Anomaly"], _test_pdf[TARGET].value_counts().sort_index(), color=["#43A047", "#E53935"])
axes[1, 1].set_title("Test target distribution (post-IF)")

_f1_if = f1_score(_test_pdf[TARGET], _test_pdf["signal_isolation_forest"], zero_division=0)
axes[1, 2].text(0.5, 0.5, f"IF F1 vs ensemble (test): {_f1_if:.3f}", ha="center", va="center", fontsize=14)
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_if_diagnostics.png"), dpi=100)
plt.show()
with mlflow.start_run(run_name="pca_if_diagnostics"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "spark_pca_anomaly"})
    mlflow.log_metric("if_f1_vs_ensemble", _f1_if)
    mlflow.log_artifact(os.path.join(OUT_DIR, "pca_if_diagnostics.png"))


In [ ]:
#  CELL 16 : Station-level aggregation + descriptive analytics
STATION_COL = next((c for c in df_spark.columns if any(
    k in c.lower() for k in ["facility_id", "facility", "station", "stop", "location"]
)), None)
_SEP = "=" * 65

print(_SEP)
print("PS4 — Dataset Overview")
print(_SEP)
print(f"  Total rows: {df_spark.count():,}")
_rng = df_spark.agg(F.min("hour_dt"), F.max("hour_dt")).collect()[0]
print(f"  Date range: {_rng[0]} → {_rng[1]}")
_tr = df_spark.agg(F.sum(TARGET).alias("pos"), F.mean(TARGET).alias("rate")).collect()[0]
print(f"  Target: {int(_tr['pos'] or 0):,} anomaly hours ({_tr['rate']:.2%})")
print(f"  Features: {len(FEATURE_COLS)}  Train/val/test: {train_sdf.count():,} / {val_sdf.count():,} / {test_sdf.count():,}")

# Anomaly by device type chart
if DEV_COL in df_spark.columns:
    _dev_anom = (
        df_spark.groupBy(DEV_COL)
        .agg(F.mean(TARGET).alias("anomaly_rate"), F.count("*").alias("total_hours"), F.sum(TARGET).alias("anomaly_hours"))
        .orderBy(F.desc("anomaly_rate")).toPandas()
    )
    print("\nAnomaly rate by device type:")
    print(_dev_anom.round(4).to_string(index=False))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(_dev_anom[DEV_COL], _dev_anom["anomaly_rate"] * 100, color="#1565C0")
    ax.set_title("Anomaly Rate (%) by Device Type", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "device_anomaly_rate.png"), dpi=100)
    plt.show()
    with mlflow.start_run(run_name="anomaly_rate_by_device"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        mlflow.log_artifact(os.path.join(OUT_DIR, "device_anomaly_rate.png"))

# Monthly trend with split markers
_monthly = (
    df_spark.withColumn("month", F.date_format("hour_dt", "yyyy-MM"))
    .groupBy("month", "split").agg(F.mean(TARGET).alias("rate"), F.count("*").alias("hours"))
    .orderBy("month").toPandas()
)
fig, ax = plt.subplots(figsize=(12, 4))
for _sp, _col in [("train", "#1565C0"), ("val", "#F59E0B"), ("test", "#E53935")]:
    _m = _monthly[_monthly["split"] == _sp]
    if len(_m):
        ax.plot(_m["month"], _m["rate"] * 100, marker="o", label=_sp, color=_col)
ax.set_title("Monthly anomaly rate by split", fontweight="bold")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "monthly_anomaly_trend.png"), dpi=100)
plt.show()
with mlflow.start_run(run_name="monthly_anomaly_trend"):
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact(os.path.join(OUT_DIR, "monthly_anomaly_trend.png"))

# Feature-target correlation (sample)
train_sdf = df_spark.filter(F.col("split") == "train")
_corr_cols = [c for c in FEATURE_COLS[:30] if c != TARGET]
_sample = spark_to_pandas(train_sdf, cols=_corr_cols + [TARGET], max_rows=100_000)
_feat_avail = [c for c in _corr_cols if c in _sample.columns]
_missing = [c for c in _corr_cols if c not in _sample.columns]
if _missing:
    print(f"[INFO] Correlation plot skipping {len(_missing)} cols not in train sample: {_missing[:5]}{'...' if len(_missing) > 5 else ''}")
if TARGET in _sample.columns and len(_feat_avail) >= 2:
    _corr = _sample[_feat_avail].corrwith(_sample[TARGET]).abs().sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    _corr.sort_values().plot(kind="barh", ax=ax, color="#1565C0")
    ax.set_title("Top 15 |feature-target| correlations (train sample)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "feature_target_corr.png"), dpi=100)
    plt.show()
    with mlflow.start_run(run_name="feature_target_correlation"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        mlflow.log_artifact(os.path.join(OUT_DIR, "feature_target_corr.png"))
else:
    print(f"[INFO] Skipping feature-target correlation plot ({len(_feat_avail)} features available)")

# Station aggregation
if STATION_COL:
    _st = (
        df_spark.groupBy(STATION_COL, "hour_dt")
        .agg(F.mean(TARGET).alias("station_anomaly_rate"), F.count("*").alias("n_devices"))
        .withColumn("station_fault", (F.col("station_anomaly_rate") > 0.5).cast("int"))
    )
    df_spark = df_spark.join(
        _st.select(STATION_COL, "hour_dt", "station_anomaly_rate", "station_fault"),
        on=[STATION_COL, "hour_dt"], how="left",
    )
    if "station_anomaly_rate" not in FEATURE_COLS:
        FEATURE_COLS = sorted(set(FEATURE_COLS + ["station_anomaly_rate", "station_fault"]))
    sync_splits_and_features()
    _top_st = (
        df_spark.groupBy(STATION_COL)
        .agg(F.mean(TARGET).alias("anomaly_rate"), F.count("*").alias("hours"))
        .orderBy(F.desc("anomaly_rate")).limit(15).toPandas()
    )
    print(f"\nTop 15 stations by anomaly rate ({STATION_COL}):")
    print(_top_st.round(4).to_string(index=False))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(_top_st[STATION_COL].astype(str), _top_st["anomaly_rate"] * 100, color="#E65100")
    ax.set_title("Top 15 Stations — Anomaly Rate", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "top_stations.png"), dpi=100)
    plt.show()
    with mlflow.start_run(run_name="top_stations_anomaly_rate"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
        mlflow.log_artifact(os.path.join(OUT_DIR, "top_stations.png"))
else:
    print("[INFO] No station/facility column found — station analytics skipped")


In [ ]:
#  CELL 17 : Summary dashboard
_SEP = "=" * 65
print(_SEP)
print("PS4 PySpark — Pipeline Summary")
print(_SEP)
print(f"  Rows            : {df_spark.count():,}")
print(f"  Features        : {len(FEATURE_COLS)}")
print(f"  Signals         : {SIGNAL_COLS}")
print(f"  Checkpoint      : {CKPT_DIR}  stage={CHECKPOINT_STAGE}")
print(f"  scale_pos_weight: {SCALE_POS_WEIGHT}")

print(f"\n{_SEP}\nSignal health (activation rate):")
for s in SIGNAL_COLS + [TARGET]:
    _r = df_spark.agg(F.mean(s)).collect()[0][0]
    _status = "ACTIVE" if _r >= 0.01 else "DORMANT"
    print(f"  {s:<28} {_r:6.2%}  [{_status}]")

print(f"\n{_SEP}\nSplit summary:")
for _sp, _sdf in [("train", train_sdf), ("val", val_sdf), ("test", test_sdf)]:
    _rng = _sdf.agg(F.min("hour_dt"), F.max("hour_dt")).collect()[0]
    _rate = _sdf.agg(F.mean(TARGET)).collect()[0][0]
    print(f"  {_sp:5s}: {_sdf.count():>10,} rows  {_rng[0]} → {_rng[1]}  anomaly={_rate:.2%}")



In [ ]:
#  CELL 18 : Prophet — 7-day anomaly rate forecast (Spark agg → pandas)
try:
    from prophet import Prophet
except ImportError:
    print("[SKIP] prophet not installed")
else:
    _daily = (
        df_spark.withColumn("ds", F.date_format("hour_dt", "yyyy-MM-dd"))
        .groupBy("ds", DEV_COL).agg(F.mean(TARGET).alias("y"))
        .orderBy("ds").toPandas()
    )
    _daily["ds"] = pd.to_datetime(_daily["ds"])
    _fig, _axes = plt.subplots(len(DEVICE_TYPES), 1, figsize=(12, 4 * len(DEVICE_TYPES)), squeeze=False)
    _forecast_rows = []
    for i, _dt in enumerate(DEVICE_TYPES):
        _sub = _daily[_daily[DEV_COL] == _dt][["ds", "y"]].copy()
        if len(_sub) < 30:
            print(f"[SKIP] {_dt}: insufficient history for Prophet")
            continue
        _m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
        _m.fit(_sub)
        _future = _m.make_future_dataframe(periods=7)
        _fc = _m.predict(_future)
        _ax = _axes[i, 0]
        _m.plot(_fc, ax=_ax)
        _ax.set_title(f"Prophet forecast — {_dt} daily anomaly rate", fontweight="bold")
        with mlflow.start_run(run_name=f"prophet_forecast_{_dt.lower()}"):
            mlflow.set_tags({**COMMON_TAGS, "model_type": "prophet", "device_type": _dt})
            mlflow.log_metric("mean_daily_anomaly_rate", float(_sub["y"].mean()))
        _forecast_rows.append(_dt)
    if _forecast_rows:
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "prophet_forecast.png"), dpi=100)
        plt.show()
        with mlflow.start_run(run_name="anomaly_forecast_summary"):
            mlflow.set_tags({**COMMON_TAGS, "model_type": "prophet"})
            mlflow.log_artifact(os.path.join(OUT_DIR, "prophet_forecast.png"))
        print(f"Prophet forecasts logged for: {_forecast_rows}")


In [ ]:
#  CELL 19 : Export PS4 outputs to S3 (weekly aggregates — RDS + dashboards)
# Hourly df_spark → device-week (Mon–Sun ISO week) before write.
# Also writes device_daily/ for PS1 joins that expect transit_day grain.
# Writes under s3://{ARTIFACT_BUCKET}/chicago/ps4/
ensure_spark_alive()

if not ENABLE_S3_EXPORT:
    print("[SKIP] ENABLE_S3_EXPORT=False")
else:
    PS4_S3_EXPORT_GRAIN = "device_weekly"  # CELL 19 override (weekly primary)
    _max_dt = df_spark.agg(F.max("hour_dt")).collect()[0][0]
    _asof_date = (
        pd.Timestamp(_max_dt).strftime("%Y-%m-%d")
        if _max_dt is not None
        else pd.Timestamp.utcnow().strftime("%Y-%m-%d")
    )
    _run_ts = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
    _scored_base = f"{PS4_SCORED_S3}/asof={_asof_date}"
    print(f"[S3] Export asof={_asof_date}  grain={PS4_S3_EXPORT_GRAIN}  base={PS4_S3_BASE}")

    _sig_cols = [c for c in SIGNAL_COLS if c in df_spark.columns]
    _type_parts = [
        F.when(F.col(c) == 1, F.lit(c.replace("signal_", ""))).otherwise(F.lit(None))
        for c in _sig_cols
    ]
    _type_expr = F.concat_ws("+", *_type_parts) if _type_parts else F.lit("ensemble")
    _denom = float(len(_sig_cols) or 3)
    _score = F.coalesce(
        F.col("if_score") if "if_score" in df_spark.columns else F.lit(None),
        F.col("signal_active_count").cast("double") / F.lit(_denom),
    )
    _sev_rank = (
        F.when(F.col("signal_active_count") >= 3, F.lit(3))
        .when(F.col("signal_active_count") >= 2, F.lit(2))
        .otherwise(F.lit(1))
    )
    _x_metric = (
        F.coalesce(F.col("event_count_hour"), F.lit(0)).cast("double")
        if "event_count_hour" in df_spark.columns
        else F.lit(0.0)
    )

    # ISO week start (Monday): date_trunc('week', ...) in Spark 3+
    _week_start = F.to_date(
        F.date_trunc("week", F.col("transit_day").cast("timestamp"))
    )

    def _agg_exprs(prefix_hours_alias="total_hours"):
        aggs = [
            F.count(F.lit(1)).alias(prefix_hours_alias),
            F.sum(TARGET).alias("anomaly_hours"),
            F.max(TARGET).alias("is_anomaly_period"),
            F.max(F.col("signal_active_count").cast("int")).alias("signal_active_count_max"),
            F.max(_sev_rank).alias("severity_rank"),
            F.mean(_score).alias("anomaly_score_mean"),
            F.max(_score).alias("anomaly_score_max"),
            F.min(F.when(F.col(TARGET) == 1, F.col("hour_dt"))).alias("first_detected_at"),
            F.max(F.when(F.col(TARGET) == 1, F.col("hour_dt"))).alias("last_detected_at"),
            F.collect_set(_type_expr).alias("anomaly_types_set"),
            F.countDistinct("transit_day").alias("active_days"),
        ]
        if "if_score" in df_spark.columns:
            aggs.extend([
                F.mean(F.col("if_score")).alias("if_score_mean"),
                F.max(F.col("if_score")).alias("if_score_max"),
            ])
        if "event_count_hour" in df_spark.columns:
            aggs.append(F.mean(_x_metric).alias("event_count_mean"))
        return aggs

    def _decorate(frame, period_col):
        return (
            frame
            .withColumn("city_id", F.lit(CITY_CODE))
            .withColumn("device_id", F.col("DEVICE_ID"))
            .withColumn("device_type", F.upper(F.col(DEV_COL)))
            .withColumn("asof_date", F.lit(_asof_date))
            .withColumn(
                "severity",
                F.when(F.col("severity_rank") >= 3, F.lit("Critical"))
                .when(F.col("severity_rank") >= 2, F.lit("High"))
                .otherwise(F.lit("Medium")),
            )
            .withColumn(
                "anomaly_type",
                F.concat_ws("+", F.array_sort(F.col("anomaly_types_set"))),
            )
            .withColumn("is_outlier", F.col("is_anomaly_period"))
            .withColumn("is_anomaly_week", F.col("is_anomaly_period"))
            .drop("anomaly_types_set", "severity_rank", DEV_COL)
        )

    # --- device-week spine (one row per DEVICE_ID × week_start × device_type) ---
    _device_weekly = _decorate(
        (
            df_spark
            .withColumn("week_start", _week_start)
            .groupBy("DEVICE_ID", "week_start", DEV_COL)
            .agg(*_agg_exprs("total_hours"))
            .withColumn("week_end", F.date_add(F.col("week_start"), 6))
            .withColumn(
                "iso_year_week",
                F.concat_ws(
                    "-W",
                    F.year(F.col("week_start")).cast("string"),
                    F.lpad(F.weekofyear(F.col("week_start")).cast("string"), 2, "0"),
                ),
            )
            # Alias for consumers that still look for a date key
            .withColumn("transit_day", F.col("week_start"))
        ),
        "week_start",
    )
    _device_weekly_uri = f"{_scored_base}/device_weekly/"
    write_parquet_s3(_device_weekly, _device_weekly_uri)
    _weekly_n = _device_weekly.count()
    print(f"  device_weekly rows={_weekly_n:,} -> {_device_weekly_uri}")

    # --- anomalies: device-weeks with >=1 anomaly hour ---
    _anom = (
        _device_weekly.filter(F.col("anomaly_hours") > 0)
        .withColumn("anomaly_score", F.col("anomaly_score_max"))
        .withColumn(
            "detected_at",
            F.coalesce(F.col("last_detected_at"), F.col("week_start").cast("timestamp")),
        )
        .withColumn("status", F.lit("active"))
    )
    _anom_cols = [
        "city_id", "device_id", "device_type", "anomaly_type", "severity",
        "anomaly_score", "detected_at", "status", "asof_date",
        "signal_active_count_max", "anomaly_hours", "total_hours",
        "week_start", "week_end", "iso_year_week", "active_days", "transit_day",
        "first_detected_at", "last_detected_at",
    ]
    if "if_score_mean" in _anom.columns:
        _anom_cols.extend(["if_score_mean", "if_score_max"])
    _anom_out = _anom.select(*[c for c in _anom_cols if c in _anom.columns])
    _anom_uri = f"{_scored_base}/anomalies/"
    write_parquet_s3(_anom_out, _anom_uri)
    _anom_n = _anom_out.count()
    print(f"  anomalies rows={_anom_n:,} (device-weeks) -> {_anom_uri}")

    # --- timeline: weekly anomaly counts by device type ---
    _timeline = (
        _device_weekly.groupBy("week_start", "device_type")
        .agg(
            F.sum("anomaly_hours").alias("count"),
            F.sum("total_hours").alias("total_hours"),
            F.avg("anomaly_score_mean").alias("avg_score"),
            F.sum("active_days").alias("active_device_days"),
        )
        .withColumn("city_id", F.lit(CITY_CODE))
        .withColumn("bucket_time", F.col("week_start").cast("timestamp"))
        .withColumn("week_end", F.date_add(F.col("week_start"), 6))
        .withColumn("asof_date", F.lit(_asof_date))
        .withColumn("transit_day", F.col("week_start"))  # compat alias
    )
    _timeline_uri = f"{_scored_base}/timeline/"
    write_parquet_s3(_timeline, _timeline_uri)
    _timeline_n = _timeline.count()
    print(f"  timeline rows={_timeline_n:,} (week×device_type) -> {_timeline_uri}")

    # --- outliers: weekly if_score (all device-weeks) ---
    if "if_score_mean" in _device_weekly.columns:
        _outliers = (
            _device_weekly
            .withColumn("x_metric", F.coalesce(F.col("event_count_mean"), F.lit(0.0)))
            .withColumn("y_metric", F.col("if_score_mean"))
            .withColumn("z_score", F.col("if_score_mean"))
            .withColumn("recorded_at", F.col("week_start").cast("timestamp"))
            .select(
                "city_id", "device_type", "device_id", "week_start", "week_end",
                "iso_year_week", "transit_day",
                "x_metric", "y_metric", "is_outlier", "z_score", "recorded_at", "asof_date",
                "if_score_mean", "if_score_max", "anomaly_hours", "total_hours", "active_days",
            )
        )
        _out_uri = f"{_scored_base}/outliers/"
        write_parquet_s3(_outliers, _out_uri)
        _out_n = _outliers.count()
        print(f"  outliers rows={_out_n:,} (device-weeks) -> {_out_uri}")
    else:
        _out_uri = None
        _out_n = 0
        print("  [INFO] outliers skipped (if_score missing)")

    # --- device_daily retained for PS1 feature joins (DEVICE_ID × transit_day) ---
    _device_daily = _decorate(
        (
            df_spark.groupBy("DEVICE_ID", "transit_day", DEV_COL)
            .agg(*_agg_exprs("total_hours"))
            .withColumn("is_anomaly_day", F.col("is_anomaly_period"))
        ),
        "transit_day",
    ).drop("is_anomaly_week")
    _device_daily_uri = f"{_scored_base}/device_daily/"
    write_parquet_s3(_device_daily, _device_daily_uri)
    _daily_n = _device_daily.count()
    print(f"  device_daily rows={_daily_n:,} (kept for PS1 joins) -> {_device_daily_uri}")

    _hourly_n = int(df_spark.count())
    _manifest = {
        "ps": "PS4",
        "city": CITY_CODE,
        "asof_date": _asof_date,
        "run_ts": _run_ts,
        "export_grain": PS4_S3_EXPORT_GRAIN,
        "week_definition": "ISO week, Monday start (Spark date_trunc week)",
        "pipeline": COMMON_TAGS.get("pipeline"),
        "paths": {
            "base": PS4_S3_BASE,
            "device_weekly": _device_weekly_uri,
            "device_daily": _device_daily_uri,
            "anomalies": _anom_uri,
            "timeline": _timeline_uri,
            "outliers": _out_uri,
            "artifacts": f"{PS4_ARTIFACTS_S3}/run_date={_asof_date}/",
        },
        "counts": {
            "hourly_source_rows": _hourly_n,
            "device_weekly_rows": int(_weekly_n),
            "device_daily_rows": int(_daily_n),
            "anomaly_device_weeks": int(_anom_n),
            "timeline_buckets": int(_timeline_n),
            "outlier_device_weeks": int(_out_n),
        },
        "rds_targets": ["device_weekly", "anomalies", "anomaly_timeline", "outlier_scores"],
        "loader_hint": f"python load_ps4_scored_to_rds.py --scored {_scored_base} --asof {_asof_date} --city {CITY_CODE}",
        "ps1_join_hint": f"spark.read.parquet('{_device_daily_uri}')  # daily kept for PS1",
        "weekly_join_hint": f"spark.read.parquet('{_device_weekly_uri}')  # one row per device-week",
    }
    _manifest_local = os.path.join(OUT_DIR, f"manifest_{_asof_date}.json")
    with open(_manifest_local, "w", encoding="utf-8") as _mf:
        json.dump(_manifest, _mf, indent=2, default=str)
    upload_file_to_s3(
        _manifest_local,
        f"{PS4_S3_PREFIX}/manifest/asof={_asof_date}/manifest.json",
    )

    _uploaded = upload_out_dir_to_s3(OUT_DIR, run_date=_asof_date)
    print(f"  artifacts uploaded: {len(_uploaded)}")
    print(
        f"  compression: hourly {_hourly_n:,} -> weekly {_weekly_n:,} "
        f"({100*_weekly_n/max(_hourly_n,1):.2f}%) + daily {_daily_n:,} for PS1"
    )

    with mlflow.start_run(run_name=f"ps4_s3_export_{_asof_date}"):
        mlflow.set_tags({**COMMON_TAGS, "model_type": "s3_export", "asof_date": _asof_date})
        mlflow.log_param("asof_date", _asof_date)
        mlflow.log_param("ps4_s3_base", PS4_S3_BASE)
        mlflow.log_param("export_grain", PS4_S3_EXPORT_GRAIN)
        mlflow.log_metrics({
            "export_hourly_rows": float(_hourly_n),
            "export_device_weekly_rows": float(_weekly_n),
            "export_device_daily_rows": float(_daily_n),
            "export_anomaly_device_weeks": float(_anom_n),
            "export_timeline_rows": float(_timeline_n),
            "export_outlier_device_weeks": float(_out_n),
            "export_artifact_files": float(len(_uploaded)),
        })
        mlflow.log_artifact(_manifest_local)

    print(f"[S3] PS4 export complete — primary path: {_device_weekly_uri}")
